# From Seeing to Predicting: PLS Regression, Latent Variables, and Honest Error
### 6.5 — PLS Regression: Quantitative Prediction from Spectra

PCA (6.3) showed us the structure in a set of spectra, and its diagnostics (6.4)
told us which samples to trust. But neither *predicts a number*. The everyday
analytical question — "what is the concentration of my analyte in this sample?" —
needs a model that maps a spectrum to a property. That model is **Partial Least
Squares (PLS) regression**, and it is the workhorse of quantitative spectroscopy.

This notebook builds PLS the way the rest of the series builds everything: from
the ground up, with the chemistry visible. We cover:

- **why PCA is not enough for prediction** — its components chase *variance*, which
  is not the same as *relevance* to your property;
- **latent variables** — directions chosen for their **covariance with the
  target**, not just their spread;
- **PLS from scratch** (the NIPALS algorithm), one latent variable at a time;
- **choosing the number of latent variables** by **cross-validation**;
- **overfitting** — how training error lies, and how cross-validation catches it;
- **interpreting the regression vector** as analyte chemistry.

Because the spectra are simulated, we know the **true concentration** behind every
sample and the **true analyte spectrum**, so we grade predictions against real
values and check that the model learned the right chemistry — not a lucky
correlation.

> **The one idea to hold onto.** PCA asks *"what varies?"*; PLS asks *"what varies
> **with my property**?"* That difference is everything: a huge interferent can
> dominate the variance (and PCA) while your analyte hides in a minor component.
> PLS aims its latent variables at the property from the start. The price is that
> you must choose **how many** latent variables — too few underfits, too many
> fits noise — and the only honest judge of that choice is **cross-validation**,
> never the training error.

## 1. Setup

The standard stack, plus the `pca_svd` helper from 6.3 so we can contrast PCA
with PLS head-to-head. As always, the data is simulated so its ground truth
travels with it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simulated_data.core.axes import build_axis
from simulated_data.core.peaks import add_peaks

EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

SEED = 0
NOISE = 0.05
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


def pca_svd(X):
    '''PCA via SVD of mean-centered data (from 6.3): scores, loadings, evr.'''
    mean = X.mean(axis=0)
    U, S, Vt = np.linalg.svd(X - mean, full_matrices=False)
    return U * S, Vt, S**2 / np.sum(S**2), mean


def rmse(a, b):
    '''Root-mean-square error between two arrays.'''
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

## 2. A dataset with a known property — and a dominant interference

We simulate a realistic quantitation problem: a **target analyte** whose
concentration we want to predict, measured against **two interfering components**
whose concentrations vary much more widely. Every spectrum is

$$ x_i = y_i\,(\text{analyte}) + c_{1i}\,(\text{interferent 1}) + c_{2i}\,(\text{interferent 2}) + \text{noise}, $$

where `y` is the property of interest (the ground truth). The interferents swing
over a far larger range than the analyte, so — as in real samples — most of the
*variance* in the data has nothing to do with what we want to measure. We make
two independent sets: a **calibration** set to train on and an untouched **test**
set to judge honestly.

In [ ]:
x = build_axis(600.0, 1800.0, n_points=400)

# Pure component spectra (center, FWHM, amplitude). The analyte is modest and only
# partly distinct; the interferents are large and broad -> they dominate variance.
analyte      = add_peaks(x, [(1050.0, 25.0, 1.0), (1480.0, 30.0, 0.5)])
interferent1 = add_peaks(x, [(800.0, 60.0, 1.0), (1400.0, 80.0, 0.8)])
interferent2 = add_peaks(x, [(1250.0, 55.0, 1.0), (1650.0, 70.0, 0.9)])


def make_set(n_samples, seed):
    '''Mixtures with a KNOWN analyte concentration y plus two larger interferents.

    Returns spectra X and the true property y. Interferent concentrations span a
    much wider range than y, so they (not the analyte) drive the variance.
    '''
    rng = np.random.default_rng(seed)
    y = rng.uniform(0.2, 1.0, n_samples)                 # TARGET property (ground truth)
    c1 = rng.uniform(0.2, 2.0, n_samples)                # interferents vary widely
    c2 = rng.uniform(0.2, 2.0, n_samples)
    X = (y[:, None] * analyte
         + c1[:, None] * interferent1
         + c2[:, None] * interferent2
         + rng.normal(0, NOISE, (n_samples, 400)))
    return X, y


X_cal, y_cal = make_set(60, seed=SEED)
X_test, y_test = make_set(40, seed=SEED + 1)
print("calibration:", X_cal.shape, " test:", X_test.shape)
print("analyte concentration range (y): %.2f to %.2f" % (y_cal.min(), y_cal.max()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(x, analyte, color="#d62728", lw=2, label="analyte (target)")
axes[0].plot(x, interferent1, color="#1f77b4", lw=1.5, label="interferent 1")
axes[0].plot(x, interferent2, color="#2ca02c", lw=1.5, label="interferent 2")
axes[0].set_title("Pure components: the analyte is the small one")
axes[0].set_xlabel("Raman shift (cm$^{-1}$)"); axes[0].set_ylabel("Intensity"); axes[0].legend()

sc = axes[1].scatter([], [])
for i in range(X_cal.shape[0]):
    axes[1].plot(x, X_cal[i], lw=0.6, color=plt.cm.viridis((y_cal[i]-0.2)/0.8), alpha=0.8)
axes[1].set_title("Calibration spectra (coloured by true analyte conc.)")
axes[1].set_xlabel("Raman shift (cm$^{-1}$)"); axes[1].set_ylabel("Intensity")
fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0.2, 1.0), cmap="viridis"),
             ax=axes[1], label="true analyte conc.")
fig.tight_layout(); fig.savefig(EXPORTS / "01_components_and_spectra.png", dpi=150)
plt.show()

Look at the right panel: the colour (the property we want) is almost impossible to
read off the raw spectra, because the big swings are the interferents. The analyte
signal is really there — but it is a minor contributor buried under everything
else. This is the situation PLS is built for.

## 3. Why PCA is not enough for prediction

The obvious idea is "do PCA, then regress the property on the scores" (that is
**Principal Component Regression**, PCR). But PCA orders its components by
**variance**, and here the variance is dominated by the *interferents*. So the
first PCs describe the interference, and the analyte — our target — is pushed into
a *later*, smaller component. Regression on the first few PCs therefore misses it.

In [ ]:
scores_pca, loadings_pca, evr_pca, mean_pca = pca_svd(X_cal)

print("variance explained :", np.round(evr_pca[:4], 3))
print("|correlation of each PC's scores with the true property y|:")
for a in range(4):
    r = abs(np.corrcoef(scores_pca[:, a], y_cal)[0, 1])
    print("   PC%d (%.0f%% of variance):  r = %.2f" % (a + 1, evr_pca[a]*100, r))

The first two PCs hold ~97 % of the variance but barely correlate with the
property; the analyte only shows up in **PC3**, a tiny component. A PCR model that
keeps "enough variance" (PC1–PC2) would be nearly blind to the analyte. PLS avoids
this by building its components with `y` in view from the start — as the
component-by-component comparison below shows.

In [ ]:
# Preview: PCR vs PLS prediction error as we add components (full PLS defined next
# section). PCR wastes its first components on interferents; PLS aims at y.
def pcr_predict(X_tr, y_tr, X_te, k):
    sc, ld, _, mean = pca_svd(X_tr)
    beta, *_ = np.linalg.lstsq(sc[:, :k], y_tr - y_tr.mean(), rcond=None)
    return (X_te - mean) @ ld[:k].T @ beta + y_tr.mean()

# (pls_fit / pls_predict are defined in Section 5; this cell is re-run conceptually
#  there. We import-forward by defining a thin local PLS here for the preview.)
def _pls_preview(X_tr, y_tr, X_te, k):
    from numpy.linalg import inv, norm
    Xm, ym = X_tr.mean(0), y_tr.mean()
    Xr, yr = X_tr - Xm, (y_tr - ym).reshape(-1, 1)
    p = X_tr.shape[1]; W = np.zeros((p, k)); P = np.zeros((p, k)); Q = np.zeros(k)
    for a in range(k):
        w = Xr.T @ yr; w /= norm(w); t = Xr @ w; tt = (t.T @ t).item()
        pl = (Xr.T @ t) / tt; q = ((yr.T @ t) / tt).item()
        Xr = Xr - t @ pl.T; yr = yr - q * t
        W[:, a] = w[:, 0]; P[:, a] = pl[:, 0]; Q[a] = q
    B = W @ inv(P.T @ W) @ Q.reshape(-1, 1)
    return ((X_te - Xm) @ B).ravel() + ym

ks = range(1, 7)
pcr_err = [rmse(pcr_predict(X_cal, y_cal, X_test, k), y_test) for k in ks]
pls_err = [rmse(_pls_preview(X_cal, y_cal, X_test, k), y_test) for k in ks]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(list(ks), pcr_err, "o-", color="#1f77b4", label="PCR (variance-ranked)")
ax.plot(list(ks), pls_err, "s-", color="#d62728", label="PLS (property-aimed)")
ax.set_xlabel("components used"); ax.set_ylabel("test RMSEP")
ax.set_title("PLS reaches low error with fewer components than PCR")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "02_pcr_vs_pls.png", dpi=150)
plt.show()
print("at 2 components: PCR RMSEP %.3f  vs  PLS RMSEP %.3f" % (pcr_err[1], pls_err[1]))

With two components PCR is still hopeless (it has only described the interferents),
while PLS already predicts the analyte well — because each PLS component was chosen
to track `y`. That is the whole motivation for latent variables.

## 4. Latent variables: covariance, not just variance

A **principal component** is the direction of maximum *variance* in `X`. A **PLS
latent variable** is the direction in `X` of maximum *covariance with `y`* — it
balances "explain the spectra" against "correlate with the property." Formally the
first PLS weight vector is proportional to the covariance between the channels and
the target,

$$ w \;\propto\; X^{\mathsf T} y, $$

so wavelengths that move *with the property* get the most weight. Each latent
variable extracts one such direction; we then **deflate** (subtract what it
explained) and repeat for the next. That iterative recipe is NIPALS.

## 5. PLS from scratch: the NIPALS algorithm

One latent variable at a time:

1. **weights** `w ∝ Xᵀy` (the covariance direction), normalised;
2. **scores** `t = Xw` (sample coordinates on this latent variable);
3. **X-loading** `p = Xᵀt / (tᵀt)` and **y-loading** `q = yᵀt / (tᵀt)`;
4. **deflate**: subtract `t pᵀ` from `X` and `t q` from `y`, then repeat.

The per-channel **regression coefficients** assemble from the pieces as
`B = W (PᵀW)⁻¹ q`, so a prediction is just `ŷ = (x − x̄)·B + ȳ`. Centering happens
inside the fit, which matters for leak-free cross-validation later.

In [ ]:
def pls_fit(X, y, n_components):
    '''PLS1 regression by NIPALS, built from scratch.

    Returns the per-channel regression coefficients B (shape (n_features, 1)) and
    the X/y means used for centering, plus the internal factors for inspection.
    Centering is done here on the *training* data only.
    '''
    X = np.asarray(X, float)
    y = np.asarray(y, float).reshape(-1, 1)
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xr, yr = X - x_mean, y - y_mean              # work on centered copies
    n_features = X.shape[1]

    W = np.zeros((n_features, n_components))      # weights (covariance directions)
    Pm = np.zeros((n_features, n_components))     # X-loadings
    Q = np.zeros(n_components)                    # y-loadings
    for a in range(n_components):
        w = Xr.T @ yr                            # 1) covariance of channels with y
        w /= np.linalg.norm(w)                   #    unit length
        t = Xr @ w                               # 2) scores on this latent variable
        tt = (t.T @ t).item()
        p = (Xr.T @ t) / tt                      # 3) X-loading
        q = ((yr.T @ t) / tt).item()            #    y-loading
        Xr = Xr - t @ p.T                        # 4) deflate X ...
        yr = yr - q * t                          #    ... and y
        W[:, a], Pm[:, a], Q[a] = w[:, 0], p[:, 0], q

    # Collapse the factors into one regression vector B.
    B = W @ np.linalg.inv(Pm.T @ W) @ Q.reshape(-1, 1)
    return {"B": B, "x_mean": x_mean, "y_mean": y_mean, "W": W, "P": Pm, "Q": Q}


def pls_predict(model, X):
    '''Predict y from spectra using a fitted PLS model.'''
    return ((np.asarray(X, float) - model["x_mean"]) @ model["B"]).ravel() + model["y_mean"]

## 6. A first PLS model

Fit a few latent variables, then look at **predicted vs. true** on both the
calibration set and the held-out test set. Points on the diagonal are perfect
predictions. (We deliberately do not yet optimise the number of components — that
is the next step.)

In [ ]:
model3 = pls_fit(X_cal, y_cal, n_components=3)
yhat_cal = pls_predict(model3, X_cal)
yhat_test = pls_predict(model3, X_test)

print("3-LV model:  calibration RMSE = %.4f   test RMSEP = %.4f"
      % (rmse(yhat_cal, y_cal), rmse(yhat_test, y_test)))

fig, ax = plt.subplots(figsize=(6, 6))
lims = [0.1, 1.1]
ax.plot(lims, lims, "k--", lw=1, label="perfect")
ax.scatter(y_cal, yhat_cal, s=40, color="#bbb", label="calibration")
ax.scatter(y_test, yhat_test, s=55, color="#d62728", edgecolor="white", label="test")
ax.set_xlabel("true analyte concentration"); ax.set_ylabel("PLS prediction")
ax.set_title("Predicted vs true (3 latent variables)")
ax.set_xlim(lims); ax.set_ylim(lims); ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "03_predicted_vs_true.png", dpi=150)
plt.show()

## 7. Choosing the number of latent variables: cross-validation

How many latent variables? Too few and the model can't capture the analyte
(underfitting); too many and it starts modelling noise (overfitting). The honest
way to choose is **k-fold cross-validation**: repeatedly hold out part of the
calibration set, fit on the rest, and predict the held-out part — for each
candidate number of components. The error on data the model **didn't train on**
(RMSECV) is what we minimise.

Crucially, the model (including its centering) is re-fit **inside each fold** on
the training part only — otherwise information from the held-out samples leaks in
and the error looks better than it really is.

In [ ]:
def cross_val_rmse(X, y, n_components, n_folds=5, seed=0):
    '''k-fold cross-validated RMSE for a given number of PLS components.

    The model is re-fit on each fold's training rows only (no leakage).
    '''
    idx = np.random.default_rng(seed).permutation(len(X))
    folds = [idx[i::n_folds] for i in range(n_folds)]
    resid = []
    for val in folds:
        train = np.setdiff1d(np.arange(len(X)), val)
        m = pls_fit(X[train], y[train], n_components)
        resid.append(pls_predict(m, X[val]) - y[val])
    return float(np.sqrt(np.mean(np.concatenate(resid) ** 2)))


max_lv = 8
rmsecv = [cross_val_rmse(X_cal, y_cal, k, n_folds=5, seed=SEED) for k in range(1, max_lv + 1)]
best_lv = int(np.argmin(rmsecv) + 1)
print("RMSECV by #LV:", {k: round(v, 4) for k, v in enumerate(rmsecv, 1)})
print("minimum RMSECV at %d latent variables" % best_lv)

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(range(1, max_lv + 1), rmsecv, "o-", color="#4c72b0")
ax.axvline(best_lv, color="red", ls="--", label=f"chosen #LV = {best_lv}")
ax.set_xlabel("number of latent variables"); ax.set_ylabel("RMSECV")
ax.set_title("Cross-validation picks the number of latent variables")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "04_rmsecv.png", dpi=150)
plt.show()

## 8. Overfitting: why training error lies

Watch what happens as we keep adding latent variables. The **calibration** error
falls forever — with enough components a model can fit its training data almost
perfectly. But the **cross-validated** and **test** errors bottom out at the right
number of components and then *rise* as the extra components start fitting noise.
That divergence is the signature of overfitting, and it is why you must never pick
the number of components from the training fit.

In [ ]:
train_err, test_err = [], []
for k in range(1, max_lv + 1):
    m = pls_fit(X_cal, y_cal, k)
    train_err.append(rmse(pls_predict(m, X_cal), y_cal))
    test_err.append(rmse(pls_predict(m, X_test), y_test))

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.plot(range(1, max_lv + 1), train_err, "o-", color="#55a868", label="calibration (train) RMSE")
ax.plot(range(1, max_lv + 1), rmsecv, "s-", color="#4c72b0", label="cross-validation RMSECV")
ax.plot(range(1, max_lv + 1), test_err, "^-", color="#d62728", label="test RMSEP")
ax.axvline(best_lv, color="k", ls=":", label=f"chosen #LV = {best_lv}")
ax.set_xlabel("number of latent variables"); ax.set_ylabel("RMSE")
ax.set_title("Training error keeps falling; CV/test error turns back up — overfitting")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "05_overfitting.png", dpi=150)
plt.show()

print("at chosen %d LV : train %.4f  CV %.4f  test %.4f"
      % (best_lv, train_err[best_lv-1], rmsecv[best_lv-1], test_err[best_lv-1]))
print("at %d LV        : train %.4f  CV %.4f  test %.4f  <- train looks better, CV/test worse"
      % (max_lv, train_err[-1], rmsecv[-1], test_err[-1]))

## 9. Final model: predict the test set, graded against truth

Fit the chosen model on the full calibration set and score it on the untouched
test set the way you would report it: **RMSEP** (prediction error), **R²**
(variance explained), and **bias** (systematic over/under-prediction). Because we
know the true concentrations, these are exact.

In [ ]:
final = pls_fit(X_cal, y_cal, n_components=best_lv)
pred = pls_predict(final, X_test)

rmsep = rmse(pred, y_test)
ss_res = np.sum((y_test - pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
bias = float(np.mean(pred - y_test))

print("FINAL PLS MODEL (%d latent variables), graded on the held-out test set:" % best_lv)
print("  RMSEP = %.4f   (analyte range %.2f-%.2f)" % (rmsep, y_test.min(), y_test.max()))
print("  R²    = %.4f" % r2)
print("  bias  = %+.4f" % bias)

## 10. Interpreting the regression vector

PLS is not a black box. Its **regression vector** `B` lives on the wavenumber
axis, so — exactly like a loading in 6.3 — you read it as a spectrum: the channels
with large weight are the ones the model uses to predict the property. If the model
learned the right chemistry, those weights should sit on the **analyte's** bands
and *suppress* the interferents. We can prove it, because we know the true analyte
spectrum.

In [ ]:
B = final["B"].ravel()
r_analyte = np.corrcoef(B, analyte)[0, 1]
r_int1 = np.corrcoef(B, interferent1)[0, 1]
r_int2 = np.corrcoef(B, interferent2)[0, 1]
print("correlation of the regression vector with each pure component:")
print("   analyte      : %+.3f   <- the model keys on the analyte" % r_analyte)
print("   interferent1 : %+.3f" % r_int1)
print("   interferent2 : %+.3f" % r_int2)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(x, analyte, color="#d62728", lw=2)
axes[0].set_ylabel("true analyte"); axes[0].set_title(
    "The regression vector keys on the analyte bands (and suppresses interferents)")
axes[1].plot(x, B, color="#333", lw=1.4)
axes[1].axhline(0, color="k", lw=0.5)
for c, _ in [(1050, "A"), (1480, "A")]:
    axes[1].axvline(c, color="#d62728", ls=":", lw=1, alpha=0.7)
axes[1].set_ylabel("regression weight  B"); axes[1].set_xlabel("Raman shift (cm$^{-1}$)")
fig.tight_layout(); fig.savefig(EXPORTS / "06_regression_vector.png", dpi=150)
plt.show()

The regression vector's strongest weights land on the analyte's bands at 1050 and
1480 cm⁻¹ (correlation ≈ 1.0 with the true analyte spectrum), while the
interferent bands are pushed toward zero. The model is predicting concentration
**from the right chemistry** — the interpretability that makes PLS trusted in
regulated labs, and the same "read the vector as a spectrum" habit you learned for
loadings in 6.3.

## 11. A reusable PLS toolkit

`pls_fit`, `pls_predict`, and `cross_val_rmse` are everything you need. Here is a
one-call wrapper that cross-validates, picks the number of latent variables, fits
the final model, and reports it — ready for 6.6 (validation) and beyond.

In [ ]:
def build_pls(X, y, max_lv=8, n_folds=5, seed=0):
    '''Cross-validate #LV, fit the final PLS model, and return it with diagnostics.'''
    rmsecv = [cross_val_rmse(X, y, k, n_folds, seed) for k in range(1, max_lv + 1)]
    best = int(np.argmin(rmsecv) + 1)
    model = pls_fit(X, y, best)
    return model, {"n_lv": best, "rmsecv": rmsecv[best - 1]}


model, info = build_pls(X_cal, y_cal)
print("auto-selected latent variables :", info["n_lv"])
print("RMSECV at that choice          : %.4f" % info["rmsecv"])
print("test RMSEP                     : %.4f" % rmse(pls_predict(model, X_test), y_test))

## Key Takeaways

- **PCA describes; PLS predicts.** PCA components maximise variance, which need not
  align with your property; PLS latent variables maximise **covariance with the
  target**, so they aim at the property from the first component.
- **Latent variables are property-aimed directions.** The first weight vector is
  `w ∝ Xᵀy` — channels that move with `y` get the most weight.
- **PLS is simple from scratch.** NIPALS extracts one latent variable at a time
  (weights → scores → loadings → deflate) and collapses to a regression vector.
- **Choosing #LV is the whole game.** Too few underfits, too many overfits; choose
  by **cross-validation**, never by training error.
- **Training error lies.** It falls monotonically with more components; only
  CV/test error reveals the right number and the onset of overfitting.
- **Grade on held-out data.** Report RMSEP, R², and bias against true values on a
  test set the model never saw.
- **The regression vector is interpretable.** Read it as a spectrum: it should key
  on the analyte's bands and suppress interferents — which, with ground truth, we
  confirmed (r ≈ 1.0 with the true analyte).

## Practical Checklist

1. **Screen first.** Remove outliers/artifacts (6.4) and preprocess (4.3/4.5)
   before calibration — PLS will happily model junk variance.
2. **Split honestly.** Keep an independent test set the model never sees during
   fitting *or* component selection.
3. **Cross-validate #LV** and pick the minimum RMSECV (or the most parsimonious
   model within one standard error of it).
4. **Plot the three error curves** (train, CV, test) and confirm CV/test turn up
   after the chosen number — your overfitting check.
5. **Report RMSEP, R², and bias** on the held-out set, in the property's units.
6. **Read the regression vector** and confirm it keys on real analyte bands, not
   interferents or noise.
7. **Re-validate** whenever instrument, sample matrix, or preprocessing changes.

## Common Mistakes

- **Data leakage in cross-validation.** Centering, scaling, or scatter-correcting
  on the *full* set before CV lets held-out information leak in and flatters the
  error. Re-fit every preprocessing step inside each fold (we centre inside
  `pls_fit`). This theme returns in 7.2.
- **Choosing #LV from the training fit.** Training error always improves with more
  components; that is not evidence the model is better.
- **No independent test set.** Reporting only cross-validation, or worse only
  calibration fit, overstates real-world accuracy.
- **Too many latent variables.** The classic overfit — great on calibration,
  worse on new samples.
- **Ignoring the regression vector.** A model that predicts well but keys on an
  interferent or a noise region will fail when that nuisance changes.
- **Skipping outlier screening.** A single high-leverage bad sample (6.4) can
  distort the whole calibration.
- **Extrapolating beyond the calibration range.** Predictions outside the trained
  concentration range are unsupported.

## Reporting Guidance

- **State the preprocessing and screening** applied before calibration, and that
  every step was re-fit inside cross-validation folds.
- **Report the number of latent variables** and how it was chosen (RMSECV curve,
  one-standard-error rule).
- **Give RMSEP, R², and bias** on an independent test set, in the property's real
  units, alongside the calibration range (note that predictions outside it are
  extrapolation).
- **Show predicted-vs-true** and the **error-vs-#LV** curves so reviewers can see
  the model is not overfit.
- **Show and interpret the regression vector**, confirming it keys on analyte
  bands.
- **Describe the validation scheme** (fold count, independence of the test set) so
  the result is reproducible.

## Next Lesson

**6.6 — Validation Done Right: Cross-Validation, RMSEP, and Overfitting.** We used
cross-validation here to pick the number of latent variables; next we treat
validation as a discipline in its own right — the kinds of cross-validation and
when each is honest, test-set design, the standard error metrics (RMSEC / RMSECV /
RMSEP) and what their gaps tell you, and the subtle ways validation gets
accidentally optimistic (the data-leakage trap flagged above, explored in depth).
From there, **6.7 — Calibration Transfer** tackles moving a model between
instruments. The PLS model you built here is the thing those lessons learn to
trust — or distrust — rigorously.